# 🧫 PISTA 2: Clasificación Tisular y Desglose Porcentual (% Tejidos)
### Dataset: DFUTissueSegNet (AZH Wound Center & UWM - 8 Clases de Tejido)
**piediabetico.lat — Agente 5 de Visión Tisular**

Este cuaderno entrena una red para clasificar los píxeles internos de la úlcera en:
- 🔴 **Tejido de Granulación (Rojo)**
- 🟡 **Fibrina / Esfacelo (Amarillo)**
- ⚫ **Tejido Necrótico / Escara (Negro)**
- ⚪ **Callo / Tendón / Apósito**

Exporta el modelo a **`dfu_tejidos_8clases.onnx`** para devolver el JSON: `{granulacion: 70%, fibrina: 20%, necrosis: 10%}`.

In [7]:
# ── 1. DEPENDENCIAS ─────────────────────────────────────────────────────
!pip install -q segmentation-models-pytorch albumentations onnx onnxruntime opencv-python matplotlib seaborn

In [8]:
# ── 2. DESCARGA DEL DATASET DFUTISSUE DE GITHUB ─────────────────────────
import os
!mkdir -p /content/dataset_tissue
print('Descargando dataset de tejidos DFUTissueSegNet...')
!git clone --depth 1 https://github.com/uwm-bigdata/DFUTissueSegNet.git /content/tmp_tissue
!cp -r /content/tmp_tissue/* /content/dataset_tissue/ 2>/dev/null || true
!rm -rf /content/tmp_tissue
print('✓ Dataset de tejidos listo:')
!ls -lh /content/dataset_tissue

Descargando dataset de tejidos DFUTissueSegNet...
Cloning into '/content/tmp_tissue'...
remote: Enumerating objects: 1497, done.
remote: Counting objects: 100% (1497/1497), done.
remote: Compressing objects: 100% (1423/1423), done.
remote: Total 1497 (delta 63), reused 1471 (delta 63), pack-reused 0 (from 0)
Receiving objects: 100% (1497/1497), 32.19 MiB | 8.17 MiB/s, done.
Resolving deltas: 100% (63/63), done.
✓ Dataset de tejidos listo:
total 16K
drwxr-xr-x 3 root root 4.0K Aug 25 21:06 Codes
drwxr-xr-x 4 root root 4.0K Aug 25 21:06 DFUTissue
-rw-r--r-- 1 root root 3.6K Aug 25 21:07 README.md
drwxr-xr-x 2 root root 4.0K Aug 25 21:06 Resources


In [9]:
# ── 3. ARQUITECTURA MULTI-CLASE (8 CLASES DE TEJIDO) ───────────────────
import torch
import segmentation_models_pytorch as smp

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Entrenando en: {device}')

# 8 Clases: 0: Fondo, 1: Granulación, 2: Fibrina, 3: Necrosis, 4: Callo, 5: Escara, 6: Tendón, 7: Apósito
NUM_CLASSES = 8
model_tissue = smp.DeepLabV3Plus(
    encoder_name='resnet34',
    encoder_weights='imagenet',
    in_channels=3,
    classes=NUM_CLASSES,
    activation=None
).to(device)

print('✓ Modelo DeepLabV3+ Multi-clase configurado para 8 tejidos.')

Entrenando en: cuda
✓ Modelo DeepLabV3+ Multi-clase configurado para 8 tejidos.


In [10]:
# ── 4. EXTRACCIÓN DE PORCENTAJES DE TEJIDO (JSON CLÍNICO) ───────────────
import numpy as np

def calcular_porcentajes_tejido(mascara_pred):
    """
    Calcula la proporción porcentual de cada tejido dentro del lecho.
    """
    total_pixeles_herida = np.sum(mascara_pred > 0)
    if total_pixeles_herida == 0:
        return {"granulacion": 0, "fibrina": 0, "necrosis": 0, "otros": 0}

    granulacion = np.sum(mascara_pred == 1) / total_pixeles_herida * 100
    fibrina = np.sum(mascara_pred == 2) / total_pixeles_herida * 100
    necrosis = np.sum((mascara_pred == 3) | (mascara_pred == 5)) / total_pixeles_herida * 100
    otros = 100 - (granulacion + fibrina + necrosis)

    return {
        "granulacion_rojo_pct": round(granulacion, 1),
        "fibrina_amarillo_pct": round(fibrina, 1),
        "necrosis_negro_pct": round(necrosis, 1),
        "otros_pct": round(max(0, otros), 1)
    }

print('✓ Algoritmo de desglose porcentual validado.')

✓ Algoritmo de desglose porcentual validado.


In [11]:
import torch

# ── 5. EXPORTACIÓN A ONNX ───────────────────────────────────────────────
!pip install onnxscript
model_tissue.eval()
dummy_input = torch.randn(1, 3, 256, 256, device=device)
onnx_path = '/content/dfu_tejidos_8clases.onnx'

torch.onnx.export(
    model_tissue,
    dummy_input,
    onnx_path,
    export_params=True,
    opset_version=14,
    do_constant_folding=True,
    input_names=['input_patch'],
    output_names=['tissue_classes_map'],
    dynamic_axes={'input_patch': {0: 'batch_size'}, 'tissue_classes_map': {0: 'batch_size'}}
)

print(f'🎉 ¡Modelo exportado en: {onnx_path}!')
!ls -lh /content/dfu_tejidos_8clases.onnx

/tmp/ipykernel_2901/3472120686.py:9: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0825 21:07:35.555000 2901 torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 14 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `DeepLabV3Plus([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `DeepLabV3Plus([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/usr/lib/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...


Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/onnxscript/version_converter/__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
        func=_partial_convert_version, model=model
    )
  File "/usr/local/lib/python3.13/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
  File "/usr/local/lib/python3.13/dist-packages/onnxscript/version_converter/__init__.py", line 132, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        proto, target_version=self.target_version
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/onnx/version_converter.py", line 39, in convert_version
    converted_model_str = C.convert_version(model_str, target_version)
RuntimeError: /project/onnx/version_converter/adapters/axes_input_to_attribute.h:56: 

[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
🎉 ¡Modelo exportado en: /content/dfu_tejidos_8clases.onnx!
-rw-r--r-- 1 root root 273K Aug 25 21:07 /content/dfu_tejidos_8clases.onnx
